# Paso 1

#### Librerias y lectura

In [32]:
import json
import re
import pandas as pd
import unicodedata
from thefuzz import fuzz

In [33]:
df = pd.read_json('catalogo_tp3.jsonl', lines=True)

#### Vista inicial

In [34]:
total_inicial = len(df)
print(f"Total de registros iniciales: {total_inicial}")

Total de registros iniciales: 8747


In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8747 entries, 0 to 8746
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   source_url      8747 non-null   str                
 1   slug            8747 non-null   str                
 2   title           8747 non-null   str                
 3   authors         8747 non-null   object             
 4   genres          8747 non-null   object             
 5   seed_genre      8747 non-null   str                
 6   series          2316 non-null   str                
 7   series_number   2298 non-null   float64            
 8   cover_url       8747 non-null   str                
 9   synopsis        8746 non-null   str                
 10  synopsis_chars  8747 non-null   int64              
 11  scraped_at      8747 non-null   datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), float64(1), int64(1), object(2), str(7)
memory usage: 820.2+ KB


In [36]:
df.head()

,source_url,slug,title,authors,genres,seed_genre,series,series_number,cover_url,synopsis,synopsis_chars,scraped_at
0,https://ww3.lectulandia.com/book/de-la-caverna...,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],"[Arqueología, Divulgación, Ensayo]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eudald%20C...,La Humanidad ha entrado en una nueva fase evol...,998,2026-10-17 23:11:56+00:00
1,https://ww3.lectulandia.com/book/quienes-somos...,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],"[Arqueología, Biología, Ciencias naturales, En...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/David%20Re...,La extracción de ADN de los huesos de nuestros...,1285,2026-10-17 23:11:58+00:00
2,https://ww3.lectulandia.com/book/teotihuacan/,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],"[Arqueología, Arte, Ciencias sociales, Ensayo,...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eduardo%20...,Teotihuacan es la primera gran ciudad del cent...,1070,2026-10-17 23:12:01+00:00
3,https://ww3.lectulandia.com/book/tiro-y-las-co...,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],"[Arqueología, Economía, Ensayo, Historia, Polí...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/M%20Eugeni...,Entre los siglos VIII y VI a. C. los fenicios ...,1875,2026-10-17 23:12:03+00:00
4,https://ww3.lectulandia.com/book/la-maldicion-...,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],"[Arqueología, Ensayo, Psicología, Sociología]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Emiliano%2...,Las raíces evolutivas del sufrimiento humano\n...,1613,2026-10-17 23:12:05+00:00


#### Analisis de sinopsis

In [37]:
(df['synopsis'] == "").sum()

np.int64(0)

No hay sinopsis vacias

In [38]:
# detectar filas con sinapsis = a NaN
df[df['synopsis'].isnull()]

,source_url,slug,title,authors,genres,seed_genre,series,series_number,cover_url,synopsis,synopsis_chars,scraped_at
7969,https://ww3.lectulandia.com/book/calvario/,calvario,Calvario,[Alejandro Sawa],"[Drama, Teatro]",teatro,NaN,NaN,https://assets.lectulandia.com/b/ab/Alejandro%...,NaN,0,2026-10-18 22:48:58+00:00


Hay una sola fila con sinapsis vacia

In [39]:
df = df.dropna(subset=["synopsis"])
df = df[df["synopsis"].astype(str).str.strip() != ""]

In [40]:
# detectar cuantas y eliminar las filas con slug duplicados
df.duplicated(subset=['slug']).sum()

np.int64(0)

No hay libros duplicados

In [41]:
df = df.drop_duplicates(subset=["slug"], keep="first")

In [42]:
len(df)

8746

In [43]:
# Normalizar espacios en blanco (reemplazar saltos de línea y tabs por un único espacio)
antes = len(df)
df["synopsis"] = df["synopsis"].astype(str).apply(lambda x: re.sub(r"\s+", " ", x).strip())
df["title"] = df["title"].astype(str).apply(lambda x: re.sub(r"\s+", " ", x).strip())

In [44]:
# Contar duplicados exactos de sinopsis
sinopsis_duplicadas = df[df.duplicated(subset=["synopsis"], keep=False)]
print(f"Registros con sinopsis iguales: {len(sinopsis_duplicadas)}")

Registros con sinopsis iguales: 131


In [45]:
# Conservar una sola versión por sinopsis
df = df.drop_duplicates(subset=["synopsis"], keep="first")
eliminados_sinopsis = antes - len(df)
print(f"Libros eliminados por sinopsis iguales: {eliminados_sinopsis}")

Libros eliminados por sinopsis iguales: 98


#### Busqueda de libros "casi iguales"

Normalizacion de los titulos

In [46]:
def normalizar_titulo(texto):
    if pd.isna(texto):
        return ""

    texto = texto.lower()

    # Quitar acentos
    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

    # Eliminar puntuación
    texto = re.sub(r"[^\w\s]", " ", texto)

    # Espacios múltiples
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


df["title_norm"] = df["title"].apply(normalizar_titulo)

Buscar candidatos

In [47]:
# Columna auxiliar para agrupar por autor
df["authors_norm"] = df["authors"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else str(x)
)

pares = []

romanos = r"\b(i|ii|iii|iv|v|vi|vii|viii|ix|x)\b"

for autor, grupo in df.groupby("authors_norm"):

    grupo = grupo.reset_index()

    if len(grupo) < 2:
        continue

    for i in range(len(grupo)):
        for j in range(i + 1, len(grupo)):

            titulo1 = grupo.loc[i, "title_norm"]
            titulo2 = grupo.loc[j, "title_norm"]

            score = fuzz.token_set_ratio(titulo1, titulo2)

            if score < 90:
                continue

            # Si tienen números distintos (años, tomos, etc.)
            nums1 = re.findall(r"\d+", titulo1)
            nums2 = re.findall(r"\d+", titulo2)

            if nums1 != nums2:
                continue

            # Si cambian los números romanos
            rom1 = re.findall(romanos, titulo1)
            rom2 = re.findall(romanos, titulo2)

            if rom1 != rom2:
                continue

            # Evitar tomos o volúmenes diferentes
            palabras = ["vol", "volumen", "tomo", "parte", "libro", "numero"]

            if any(p in titulo1 or p in titulo2 for p in palabras):
                if titulo1 != titulo2:
                    continue

            pares.append({
                "idx1": grupo.loc[i, "index"],
                "idx2": grupo.loc[j, "index"],
                "autor": autor,
                "titulo1": grupo.loc[i, "title"],
                "titulo2": grupo.loc[j, "title"],
                "score_titulo": score
            })

pares = pd.DataFrame(pares)

print(f"Pares candidatos encontrados: {len(pares)}")
display(pares.head())

Pares candidatos encontrados: 110


,idx1,idx2,autor,titulo1,titulo2,score_titulo
0,451,7950,"Adolfo Bioy Casares, Jorge Luis Borges",Nuevos cuentos de Bustos Domecq,Nuevos cuentos de Bustos Domecq,100
1,490,2090,Alejandro Fabbri,Historias negras del fútbol argentino,Nuevas historias negras del fútbol argentino,100
2,529,2857,Ambrose Bierce,El diccionario del Diablo (Trad. Eduardo Stilman),El diccionario del Diablo,100
3,2857,3590,Ambrose Bierce,El diccionario del Diablo,El diccionario del diablo (trad. Rodolfo Walsh),100
4,1603,7955,Anónimo,Lazarillo de Tormes (ed. Francisco Rico),Lazarillo de Tormes,100


Comparar sinopsis

In [48]:
pares["score_sinopsis"] = pares.apply(
    lambda fila: fuzz.token_set_ratio(
        df.loc[fila["idx1"], "synopsis"],
        df.loc[fila["idx2"], "synopsis"]
    ),
    axis=1
)

display(pares.head())

,idx1,idx2,autor,titulo1,titulo2,score_titulo,score_sinopsis
0,451,7950,"Adolfo Bioy Casares, Jorge Luis Borges",Nuevos cuentos de Bustos Domecq,Nuevos cuentos de Bustos Domecq,100,59
1,490,2090,Alejandro Fabbri,Historias negras del fútbol argentino,Nuevas historias negras del fútbol argentino,100,58
2,529,2857,Ambrose Bierce,El diccionario del Diablo (Trad. Eduardo Stilman),El diccionario del Diablo,100,51
3,2857,3590,Ambrose Bierce,El diccionario del Diablo,El diccionario del diablo (trad. Rodolfo Walsh),100,58
4,1603,7955,Anónimo,Lazarillo de Tormes (ed. Francisco Rico),Lazarillo de Tormes,100,59


Elegir las posibles "obras iguales" y mostrarlas

In [49]:
obras_iguales = pares[
    pares["score_sinopsis"] >= 80
].sort_values(
    ["score_titulo", "score_sinopsis"],
    ascending=False
)

print(f"Posibles obras iguales: {len(obras_iguales)}")

Posibles obras iguales: 25


Mostramos estas obras

In [50]:
display(
    obras_iguales[
        [
            "idx1",
            "idx2",
            "autor",
            "titulo1",
            "titulo2",
            "score_titulo",
            "score_sinopsis"
        ]
    ]
)

,idx1,idx2,autor,titulo1,titulo2,score_titulo,score_sinopsis
7,1309,1504,Arthur Conan Doyle,Sherlock Holmes,Sherlock Holmes. La colección completa,100,100
37,4862,5876,H. Rider Haggard,"Ella, Hija de la Sabiduría",Ella. Hija de la sabiduría,100,100
43,206,6001,Isaac Asimov,El secreto del universo,El secreto del universo y otros ensayos cientí...,100,100
53,208,6218,John Gribbin,Introducción a la ciencia. Una guía para todos...,Introducción a la ciencia,100,100
61,1550,1803,Lewis Wallace,Ben-Hur,Ben-Hur (Traducción Raimundo Griñó),100,100
64,2065,8462,Mark Twain,El príncipe y el mendigo,Príncipe y mendigo,100,100
65,736,739,Maurice Nicoll,Comentarios psicológicos sobre las enseñanzas ...,Comentarios psicológicos sobre las enseñanzas ...,100,100
82,8028,8032,Richard Ford,Manual para viajeros por Andalucía y lectores ...,Manual para viajeros por Andalucía y lectores ...,100,100
49,359,7456,Jean-Jacques Rousseau,El contrato social,Contrato social,100,99
60,801,5290,Laura Gallego García,La emperatriz de los Etéreos (Ed. Ilustrada),La emperatriz de los Etéreos,100,99


# PASO 2

## 2.1

In [51]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, confusion_matrix

# Asumiendo que 'muestra_etiquetada.csv' tiene las columnas 'synopsis' y 'contaminado'
df_muestra = pd.read_csv('muestra_etiquetada.csv')

# Eliminar posibles nulos en la sinopsis para evitar errores con regex
df_muestra = df_muestra.dropna(subset=['synopsis'])

# División del dataset (70% desarrollo, 30% prueba)
X_train, X_test, y_train, y_test = train_test_split(
    df_muestra['synopsis'], 
    df_muestra['contaminado'], 
    test_size=0.30, 
    random_state=42, 
    stratify=df_muestra['contaminado']
)

print(f"Tamaño de entrenamiento: {len(X_train)}")
print(f"Tamaño de prueba: {len(X_test)}")

Tamaño de entrenamiento: 350
Tamaño de prueba: 150


In [52]:
df_muestra.head()

,slug,title,synopsis,contaminado,fragmento
0,el-cinturon-de-venus,El cinturón de Venus,"La parábola, la intriga y el humor, tan escaso...",0,NaN
1,los-hijos-de-la-llanura-2,Los hijos de la llanura,Una historia apasionante que te atrapará desde...,1,Una historia apasionante que te atrapará desde...
2,electra,Electra,Entre las siete tragedias de Sófocles (c. 496 ...,0,NaN
3,el-despertar-de-la-senorita-prim,El despertar de la señorita Prim,Atraída por un sugestivo anuncio en un periódi...,0,NaN
4,comprometer-a-un-vizconde,Comprometer a un Vizconde,"Su vida es el ejército, no tiene tiempo para e...",0,NaN


In [53]:
# Celda de código: Expresión regular y evaluación
# Regex para URLs (http/https o www), Correos y Teléfonos (formatos internacionales y locales básicos)
patron_contaminacion = re.compile(
    r'(https?://\S+|www\.\S+)' # URLs
    r'|([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})' # Correos
    r'|(\+?\d[\d\s\-\(\)]{7,15}\d)', # Teléfonos
    re.IGNORECASE
)

# Función para predecir si está contaminado basándose en la regex
def predecir_con_regex(texto):
    if patron_contaminacion.search(str(texto)):
        return 1
    return 0

# Aplicar el detector al conjunto de prueba
y_pred = X_test.apply(predecir_con_regex)

# Medir Precisión y Recall
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f"Precisión (Precision): {precision:.4f}")
print(f"Exhaustividad (Recall): {recall:.4f}")

# Explorar falsos positivos
falsos_positivos = X_test[(y_pred == 1) & (y_test == 0)]
print(f"\nCantidad de Falsos Positivos: {len(falsos_positivos)}")
# print(falsos_positivos.head()) # Descomentar para leer ejemplos

Precisión (Precision): 0.6364
Exhaustividad (Recall): 0.2333

Cantidad de Falsos Positivos: 4


## 2.2

In [54]:
import unicodedata
import re
from sklearn.metrics import precision_score, recall_score

def normalizar(texto):
    if not isinstance(texto, str):
        return ""
    
    # 1. Normalizar caracteres de otros alfabetos/anchuras (ej. ｗ -> w, ａ -> a)
    texto = unicodedata.normalize('NFKC', texto)
    
    # 2. Filtrar caracteres invisibles o de control (Categorías Cf y Cc, excepto saltos de línea/tabs)
    texto = "".join(
        c for c in texto 
        if unicodedata.category(c) not in ['Cf', 'Cc'] or c in '\n\r\t'
    )
    
    # 3. Reemplazar palabras usadas para disimular signos
    reemplazos = {
        r'\bpunto\b': '.',
        r'\bdot\b': '.',
        r'\barroba\b': '@',
        r'\bat\b': '@'
    }
    for patron, reemplazo in reemplazos.items():
        texto = re.sub(patron, reemplazo, texto, flags=re.IGNORECASE)
    
    # 4. Eliminar espacios estratégicos en subcadenas típicas de enlaces y correos
    texto = re.sub(r'(?i)\bw\s+w\s+w\b', 'www', texto)
    texto = re.sub(r'(?i)\bc\s+o\s+m\b', 'com', texto)
    
    # 5. Juntar signos despegados (ej. "www . misitio . com" -> "www.misitio.com")
    texto = re.sub(r'\s*\.\s*', '.', texto)
    texto = re.sub(r'\s*@\s*', '@', texto)
    
    return texto

# Aplicar la normalización al conjunto de prueba
X_test_norm = X_test.apply(normalizar)

# Volver a predecir usando la misma regex del Paso 2.1
y_pred_norm = X_test_norm.apply(predecir_con_regex)

# Medir el impacto en las métricas
precision_norm = precision_score(y_test, y_pred_norm)
recall_norm = recall_score(y_test, y_pred_norm)

print(f"Precisión (Normalizado): {precision_norm:.4f}")
print(f"Exhaustividad (Recall Normalizado): {recall_norm:.4f}")

# Identificar los casos que ANTES no se detectaban y AHORA sí (Falsos Negativos -> Verdaderos Positivos)
casos_recuperados = X_test[(y_pred == 0) & (y_pred_norm == 1) & (y_test == 1)]

print(f"\nSe recuperaron {len(casos_recuperados)} registros contaminados ocultos.")
print("\nEjemplos de casos recuperados:")
# Mostrar 2 casos recuperados
for texto_original in casos_recuperados.head(2):
    print(f"\nOriginal : {texto_original}")
    print(f"Normalizado: {normalizar(texto_original)}")

Precisión (Normalizado): 0.6923
Exhaustividad (Recall Normalizado): 0.3000

Se recuperaron 2 registros contaminados ocultos.

Ejemplos de casos recuperados:

Original : Las opiniones del nuevo Papa Francisco I sobre la familia, la fe y el papel de la Iglesia en el siglo XXI. Buscalo en w w w . novelalibre . example El Cardenal Jorge Mario Bergoglio, desde el 13 de marzo de 2013 Papa Francisco, y el Rabino Abraham Skorka, son dos tenaces promotores del diálogo interreligioso, a través del cual buscan construir horizontes comunes sin diluir las particularidades que los caracterizan. Sobre el cielo y la tierra es el resultado de una serie de profundas conversaciones que mantuvieron de manera alternada en la sede del Episcopado y en la comunidad judía Benei Tikva. En sus encuentros transitaron las más variadas cuestiones teológicas y terrenales. Dios, el fundamentalismo, los ateos, la muerte, el Holocausto, la homosexualidad, el capitalismo, son apenas un puñado de los temas en los que dan

## 2.3

In [ ]:
# Celda de código: Detección de idioma con langdetect
import nltk
from langdetect import detect, detect_langs, DetectorFactory
from nltk.tokenize import sent_tokenize

DetectorFactory.seed = 42 

def detectar_idioma_global(texto):
    """Detecta el idioma de toda la sinopsis."""
    try:
        return detect(str(texto))
    except:
        return 'unknown'

# Observación sobre la sinopsis completa en el conjunto de desarrollo (X_train)
idiomas_globales = X_train.apply(detectar_idioma_global)
# La inmensa mayoría será 'es', ocultando el spam en inglés.

In [60]:
# Celda de código: Detección por oraciones y justificación de umbrales
def contiene_idioma_extranjero(texto, min_palabras=4, umbral_prob=0.80):
    """
    Divide el texto en oraciones y busca oraciones en idiomas distintos al español.
    Se aplican umbrales de longitud y probabilidad para evitar falsos positivos.
    """
    if not isinstance(texto, str):
        return 0
        
    oraciones = sent_tokenize(texto)
    
    for oracion in oraciones:
        palabras = oracion.split()
        
        # Filtro 1: Cantidad mínima de palabras
        if len(palabras) >= min_palabras:
            try:
                predicciones = detect_langs(oracion)
                mejor_pred = predicciones[0]
                
                # Filtro 2: Umbral de probabilidad y que no sea español
                if mejor_pred.lang != 'es' and mejor_pred.prob >= umbral_prob:
                    return 1 # Contaminado
            except:
                continue
                
    return 0 # Limpio

# Aplicar el detector al conjunto de prueba
y_pred_idioma = X_test.apply(contiene_idioma_extranjero)

# Medir el rendimiento de este detector específico
precision_idioma = precision_score(y_test, y_pred_idioma)
recall_idioma = recall_score(y_test, y_pred_idioma)

print(f"Precisión (Detector de Idioma): {precision_idioma:.4f}")
print(f"Exhaustividad (Detector de Idioma): {recall_idioma:.4f}")

Precisión (Detector de Idioma): 0.5455
Exhaustividad (Detector de Idioma): 0.4000


## 2.4

In [ ]:
# Celda de código: Instalación (si es necesaria) y aplicación de GLiNER
# !pip install gliner

from gliner import GLiNER
import pandas as pd

# Cargar el modelo GLiNER (versión multilingüe o español)
modelo_gliner = GLiNER.from_pretrained("urchade/gliner_multi_pii-v1")

# Etiquetas personalizadas orientadas a buscar contaminación
etiquetas = ["persona", "organización", "lugar", "sitio web", "marca comercial", "precio", "correo electrónico"]

# Separar oraciones contaminadas y limpias (asumiendo que y_test == 1 es contaminado)
oraciones_contaminadas = X_test[y_test == 1].dropna().head(10).tolist()
oraciones_limpias = X_test[y_test == 0].dropna().head(10).tolist()

def extraer_entidades(textos, modelo, etiquetas):
    resultados = []
    for texto in textos:
        # GLiNER predice sobre el texto pasándole las etiquetas deseadas
        entidades = modelo.predict_entities(texto, etiquetas, threshold=0.5)
        for ent in entidades:
            resultados.append((ent["text"], ent["label"]))
    return resultados

entidades_contaminadas = extraer_entidades(oraciones_contaminadas, modelo_gliner, etiquetas)
entidades_limpias = extraer_entidades(oraciones_limpias, modelo_gliner, etiquetas)

print("--- Entidades en textos CONTAMINADOS ---")
for texto, etiqueta in set(entidades_contaminadas):
    print(f"[{etiqueta.upper()}] {texto}")

print("\n--- Entidades en textos LIMPIOS ---")
for texto, etiqueta in set(entidades_limpias):
    print(f"[{etiqueta.upper()}] {texto}")

c:\Users\maxir\OneDrive\Documents\trabajos nlp\ej1_grupo\Procesamiento-del-Lenguaje-Natural\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\maxir\OneDrive\Documents\trabajos nlp\ej1_grupo\Procesamiento-del-Lenguaje-Natural\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
c:\Users\maxir\OneDrive\Documents\trabajos nlp\ej1_grupo\Procesamiento-del-Lenguaje-Natural\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests t